In [4]:
import pandas as pd
import os
import requests
import time
from pathlib import Path
import json

# 配置参数
INPUT_CSV = '/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/creativity/Usefulness_Novelty.csv'
LYRIC_FOLDER = '/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/lyrics_folder'

# 创建歌词文件夹（如果不存在）
Path(LYRIC_FOLDER).mkdir(parents=True, exist_ok=True)

# 网易云音乐API配置
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f"歌词将保存到: {LYRIC_FOLDER}")

歌词将保存到: /Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/lyrics_folder


In [5]:
def get_lyric_from_netease(song_id):
    """
    从网易云音乐API获取歌词
    
    Args:
        song_id: 歌曲ID (来自CSV)
    
    Returns:
        str: 歌词文本，如果获取失败返回None
    """
    try:
        # 网易云音乐歌词API接口
        url = f'https://music.163.com/api/song/lyric?id={song_id}&lv=1&kv=1&tv=-1'
        
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'
        
        if response.status_code == 200:
            data = response.json()
            
            # 检查是否有歌词
            if data.get('lrc') and data['lrc'].get('lyric'):
                return data['lrc']['lyric']
            elif data.get('nolyric'):
                return '[纯音乐，无歌词]'
            elif data.get('uncollection'):
                return '[歌词未收录]'
            else:
                return None
        else:
            return None
            
    except Exception as e:
        print(f"  ⚠️ 错误: {str(e)}")
        return None

# 测试函数
print("✅ 歌词获取函数已定义")

✅ 歌词获取函数已定义


In [6]:
def download_all_lyrics():
    """
    从CSV文件读取所有歌曲，并下载对应的歌词
    """
    
    # 读取CSV文件
    df = pd.read_csv(INPUT_CSV)
    print(f"📊 加载了 {len(df)} 首歌曲")
    print(f"列名: {list(df.columns)[:5]}...")
    
    success = 0  # 成功计数
    failed = 0   # 失败计数
    
    # 遍历每一行
    for i, row in df.iterrows():
        artist = row['artist_name']
        song = row['song_name']
        song_id = str(int(row['song_id']))  # 确保song_id是整数
        
        # 创建文件名: 艺术家_歌曲名.txt
        filename = f"{artist}_{song}.txt"
        filepath = os.path.join(LYRIC_FOLDER, filename)
        
        # 如果文件已存在，跳过
        if os.path.exists(filepath):
            print(f"⏭️  [{i+1}/{len(df)}] {artist} - {song} (已存在)")
            success += 1
            continue
        
        # 获取歌词
        print(f"⬇️  [{i+1}/{len(df)}] 正在下载: {artist} - {song} (ID: {song_id})...", end=" ")
        lyric = get_lyric_from_netease(song_id)
        
        if lyric:
            # 保存歌词到文件
            try:
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(f"歌手: {artist}\n")
                    f.write(f"歌曲: {song}\n")
                    f.write(f"ID: {song_id}\n")
                    f.write("=" * 50 + "\n\n")
                    f.write(lyric)
                
                print("✅ 成功")
                success += 1
                
            except Exception as e:
                print(f"❌ 保存失败: {str(e)}")
                failed += 1
        else:
            print("❌ 获取失败或无歌词")
            failed += 1
        
        # 添加延迟，避免请求过于频繁
        time.sleep(0.5)
    
    # 打印统计信息
    print("\n" + "=" * 50)
    print(f"📈 下载完成!")
    print(f"✅ 成功: {success}")
    print(f"❌ 失败: {failed}")
    print(f"📂 歌词保存位置: {LYRIC_FOLDER}")
    
    return success, failed

# 运行下载
print("准备开始下载歌词...")

准备开始下载歌词...


In [7]:
# 执行下载
success_count, failed_count = download_all_lyrics()

📊 加载了 955 首歌曲
列名: ['music_genre', 'artist_name', 'song_name', 'album_name', 'release_date']...
⬇️  [1/955] 正在下载: 郑润泽 - Intro (ID: 2025185306)... ❌ 获取失败或无歌词
⬇️  [2/955] 正在下载: 郑润泽 - Outro (ID: 2089118641)... ✅ 成功
⬇️  [3/955] 正在下载: 郑润泽 - 任性 (ID: 1352916750)... ✅ 成功
⬇️  [4/955] 正在下载: 郑润泽 - 是你 (ID: 2031660746)... ✅ 成功
⬇️  [5/955] 正在下载: 郑润泽 - 只在今夜 (ID: 1352916750)... ✅ 成功
⬇️  [6/955] 正在下载: 郑润泽 - 像晴天像雨天 (ID: 1352916750)... ✅ 成功
⬇️  [7/955] 正在下载: 郑润泽 - 晚点 (ID: 1352916750)... ✅ 成功
⬇️  [8/955] 正在下载: 郑润泽 - 倔强 (ID: 1352916750)... ✅ 成功
⬇️  [9/955] 正在下载: 郑润泽 - 隐形人 (ID: 1352916750)... ✅ 成功
⬇️  [10/955] 正在下载: 郑润泽 - 一切的一切 (ID: 1352916750)... ✅ 成功
⬇️  [11/955] 正在下载: 郑润泽 - 除了你之外我没喜欢过别人 (ID: 2675204920)... ✅ 成功
⬇️  [12/955] 正在下载: 郑润泽 - 看着我 (ID: 1352916750)... ✅ 成功
⬇️  [13/955] 正在下载: 郑润泽 - Crush (ID: 1352916750)... ✅ 成功
⬇️  [14/955] 正在下载: 郑润泽 - My Dear (ID: 1352916750)... ✅ 成功
⬇️  [15/955] 正在下载: 郑润泽 - 想悄悄住进你的灵魂 (ID: 2675204921)... ✅ 成功
⬇️  [16/955] 正在下载: 郑润泽 - 雨滴中有你 (ID: 1352916750)... ✅ 成功
⬇️  [17/955] 正在

In [8]:
# 验证下载结果
import glob

lyric_files = glob.glob(os.path.join(LYRIC_FOLDER, "*.txt"))
print(f"\n📁 已保存的歌词文件: {len(lyric_files)}")
print("\n前10个文件:")
for f in sorted(lyric_files)[:10]:
    filename = os.path.basename(f)
    size = os.path.getsize(f) / 1024  # 转换为KB
    print(f"  • {filename} ({size:.1f} KB)")


📁 已保存的歌词文件: 899

前10个文件:
  • h3r3_B面唱片.txt (2.2 KB)
  • h3r3_Goodbye Goodluck.txt (3.1 KB)
  • h3r3_If This Dream.txt (2.2 KB)
  • h3r3_Magic Mirror.txt (2.2 KB)
  • h3r3_一颗麦粒.txt (2.2 KB)
  • h3r3_不寻常夏日.txt (2.2 KB)
  • h3r3_世界打了个哈欠 我们跟着失眠.txt (2.0 KB)
  • h3r3_修辞.txt (2.2 KB)
  • h3r3_原谅.txt (2.2 KB)
  • h3r3_在背后把我紧紧抱住.txt (1.2 KB)
